In [76]:
import json
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import zipfile

# **1. Load data and setup config**

In [77]:
with zipfile.ZipFile("../csv_outputs/feature_df_raw.zip") as zf:
    with zf.open("feature_df_raw.csv") as f:
        hist_feature_df = pd.read_csv(f, index_col=0)
listings_feature_df = pd.read_csv("../csv_outputs/listings_feature_df.csv")
df_rpi = pd.read_csv("../../datasets/HDBResalePriceIndex1Q2009100Quarterly.csv")
listings_predictions_json = "../json_outputs/listings_predictions.json"
ci_offsets_json = "../json_outputs/ci_offsets.json"

model_paths = {
    "lgbm": "../models/lgb_model.joblib",
    "xgb": "../models/xgb_model.ubj",
    "cb": "../models/cb_model.cbm"
}

SELECTED_MODEL = "ensemble_equal" # lgbm/xgb/cb/ensemble/ensemble_equal
ENSEMBLE_WEIGHTS = None

In [78]:
# Feature list 
FEATURES = [
    "town", "flat_type", "floor_area_sqm", "lease_commence_date",
    "remaining_lease", "lat", "lon",
    # "mall_1_name", "mall_2_name", "mall_3_name",
    "mall_1_dist_m", "mall_2_dist_m", "mall_3_dist_m",
    # "school_1_name", "school_2_name", "school_3_name",
    "school_1_dist_m", "school_2_dist_m", "school_3_dist_m",
    # "hawker_1_name", "hawker_2_name", "hawker_3_name",
    "hawker_1_dist_m", "hawker_2_dist_m", "hawker_3_dist_m",
    # "polyclinic_1_name", "polyclinic_2_name", "polyclinic_3_name",
    "polyclinic_1_dist_m", "polyclinic_2_dist_m", "polyclinic_3_dist_m",
    # "supermarket_1_name", "supermarket_2_name", "supermarket_3_name",
    "supermarket_1_dist_m", "supermarket_2_dist_m", "supermarket_3_dist_m",
    # "train_1_name", "train_2_name", "train_3_name",
    "train_1_dist_m", "train_2_dist_m", "train_3_dist_m",
    # "bus_1_name", "bus_2_name", "bus_3_name",
    "bus_1_dist_m", "bus_2_dist_m", "bus_3_dist_m",
    "num_mrt_within_1km", "flag_mrt_within_500m",
    "num_primary_schools_within_1km", "num_hawkers_within_500m",
    "num_bus_within_400m",
    "dist_cbd",
    "month_index"
]

# Columns to calculate medians for
TOWN_MEDIAN_COLS = [
    "lat", "lon", "dist_cbd",
    "mall_1_dist_m", "mall_2_dist_m", "mall_3_dist_m",
    "school_1_dist_m", "school_2_dist_m", "school_3_dist_m",
    "hawker_1_dist_m", "hawker_2_dist_m", "hawker_3_dist_m",
    "polyclinic_1_dist_m", "polyclinic_2_dist_m", "polyclinic_3_dist_m",
    "supermarket_1_dist_m", "supermarket_2_dist_m", "supermarket_3_dist_m",
    "train_1_dist_m", "train_2_dist_m", "train_3_dist_m",
    "bus_1_dist_m", "bus_2_dist_m", "bus_3_dist_m",
    "num_mrt_within_1km", "flag_mrt_within_500m",
    "num_primary_schools_within_1km", "num_hawkers_within_500m",
    "num_bus_within_400m"
]

print(f"FEATURES count: {len(FEATURES)}")

FEATURES count: 35


# **2. Build hypothetical features**

* Group by (`town`, `flat_type`) and get medians of location-based features

In [79]:
def get_median_distances(df):
    df = df.copy()
    df["town"] = df["town"].str.upper().str.strip()
    df["flat_type"] = df["flat_type"].str.upper().str.strip()

    median_cols = [c for c in TOWN_MEDIAN_COLS if c in df.columns]
    group_medians = (
        df.groupby(["town", "flat_type"])[median_cols].median().to_dict(orient="index")
    )

    print(f"[lookup] {len(group_medians)} town×flat_type entries loaded")
    return group_medians

In [80]:
# Concatenate historical and current listings df
hist_feature_df = hist_feature_df[FEATURES]
listings_feature_df = listings_feature_df[FEATURES]
combined_feature_df = pd.concat(
    [hist_feature_df, listings_feature_df],
    axis = 0
).reset_index(drop=True)

print(f"Historical rows: {len(hist_feature_df)}")
print(f"Current rows: {len(listings_feature_df)}")
print(f"Total combined: {len(combined_feature_df)}")

# Apply medians
GROUP_MEDIANS = get_median_distances(combined_feature_df)

Historical rows: 228126
Current rows: 895
Total combined: 229021
[lookup] 156 town×flat_type entries loaded


In [81]:
today = datetime.today()

def build_hypothetical_features(floor_area_sqm, town, flat_type, remaining_lease_years):
    town = town.upper().strip()
    flat_type = flat_type.upper().strip()

    ## Build distance features (using medians)
    medians = GROUP_MEDIANS.get((town, flat_type), {})

    if not medians:
        town_keys = [k for k in GROUP_MEDIANS if k[0] == town]
        if not town_keys:
            raise ValueError(f"Town {town} not found in GROUP_MEDIANS.")
        town_values = [GROUP_MEDIANS[k] for k in town_keys]
        median_cols = list(town_values[0].keys())
        medians = {
            col: float(np.nanmedian([v.get(col, np.nan) for v in town_values]))
            for col in median_cols
        }
        print(f"Warning: ({town}, {flat_type}) not in GROUP_MEDIANS."
              f"Using town-level median across {len(town_keys)} flat types.")

    row = dict(medians)

    ## Build direct inputs
    row["floor_area_sqm"] = float(floor_area_sqm)
    row["remaining_lease"] = float(remaining_lease_years)
    row["lease_commence_date"] = int(today.year + remaining_lease_years - 99)
    row["flat_type"] = flat_type
    row["town"] = town
    row["month_index"] = (today.year - 2017) * 12 + (today.month - 1)

    ## Build dataframe of new row in order
    df = pd.DataFrame([row])
    for col in FEATURES:
        if col not in df.columns:
            df[col] = np.nan
    return df[FEATURES]

## Check
sample = build_hypothetical_features(90, "ANG MO KIO", "3 ROOM", 70)
print("Feature row shape:", sample.shape)
sample

Feature row shape: (1, 35)


,town,flat_type,floor_area_sqm,lease_commence_date,remaining_lease,lat,lon,mall_1_dist_m,mall_2_dist_m,mall_3_dist_m,...,bus_1_dist_m,bus_2_dist_m,bus_3_dist_m,num_mrt_within_1km,flag_mrt_within_500m,num_primary_schools_within_1km,num_hawkers_within_500m,num_bus_within_400m,dist_cbd,month_index
0,ANG MO KIO,3 ROOM,90.0,1997,70.0,1.370398,103.848936,941.0,2215.0,2636.0,...,107.0,147.0,209.0,1.0,0.0,2.0,1.0,9.0,9639.261922,111


# **3. Predict price for hypothetical listing**
* Get current RPI
* Run models to get `predicted_price`
* Get confidence intervals of `predicted_price`: `ci_lower` and `ci_upper`

In [82]:
# RPI: get current quarter value
df_rpi.rename(columns={"index": "rpi"}, inplace=True)
df_rpi["rpi"] = pd.to_numeric(df_rpi["rpi"])
df_rpi.loc[len(df_rpi)] = { # Flash estimate from HDB
    "quarter": "2026-Q1",
    "rpi": 203.4
 }

current_quarter = f"{today.year}-Q{((today.month - 1) // 3) + 1}"

if current_quarter not in df_rpi["quarter"].values:
  print(f"Current quarter {current_quarter} not in RPI data. Extrapolating...")

  recent = df_rpi.tail(4).copy().reset_index(drop=True)
  recent["t"] = range(4)
  slope, intercept = np.polyfit(recent["t"], recent["rpi"], deg=1)

  last_quarter = df_rpi["quarter"].iloc[-1]
  lq_year, lq_num = int(last_quarter.split("-Q")[0]), int(last_quarter.split("-Q")[1])
  q_year,  q_num  = int(current_quarter.split("-Q")[0]), int(current_quarter.split("-Q")[1])
  steps_ahead = (q_year - lq_year) * 4 + (q_num - lq_num)
  t_extrap = 3 + steps_ahead
  rpi_extrap = round(intercept + slope * t_extrap, 1)

  print(f"Extrapolated RPI for {current_quarter}: {rpi_extrap:.1f} (last known: {df_rpi["rpi"].iloc[-1]:.1f}, slope: {slope:+.2f}/quarter)")
  df_rpi = pd.concat(
      [df_rpi, pd.DataFrame([{"quarter": current_quarter, "rpi": rpi_extrap}])],
      ignore_index=True
  )
else:
  val = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
  print(f"RPI for {current_quarter}: {val:.1f}")

RPI_BASE = 100.0
RPI_CURRENT = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
print(f"Base RPI: {RPI_BASE}")
print(f"Current RPI: {RPI_CURRENT:.1f} (used to convert predictions back to nominal value)")

Current quarter 2026-Q2 not in RPI data. Extrapolating...
Extrapolated RPI for 2026-Q2: 203.8 (last known: 203.4, slope: +0.14/quarter)
Base RPI: 100.0
Current RPI: 203.8 (used to convert predictions back to nominal value)


In [83]:
# Load models
def load_models(model_name):
  models = {}
  if model_name in ("lgbm", "ensemble", "ensemble_equal"):
    print("Loading LightGBM model")
    models["lgbm"] = joblib.load(model_paths["lgbm"])
  if model_name in ("xgb", "ensemble", "ensemble_equal"):
    print("Loading XGBoost model")
    m = xgb.XGBRegressor()
    m.load_model(model_paths["xgb"])
    models["xgb"] = m
  if model_name in ("cb", "ensemble", "ensemble_equal"):
    print("Loading CatBoost model")
    m = CatBoostRegressor()
    m.load_model(model_paths["cb"])
    models["cb"] = m
  return models

def predict(models, X):
  cat_cols = ["town", "flat_type"]

  def cb_predict(X):
    X_cb = X.copy()
    for col in cat_cols:
      X_cb[col] = X_cb[col].astype(str)
    pool = Pool(X_cb, cat_features=cat_cols)
    return models["cb"].predict(pool)

  if SELECTED_MODEL == "ensemble":
    w = ENSEMBLE_WEIGHTS
    if w is None:
      w = np.load("../models/ensemble_weights.npy")
    return (
        w[0] * cb_predict(X)
        + w[1] * models["xgb"].predict(X)
        + w[2] * models["lgbm"].predict(X)
    )
  if SELECTED_MODEL == "ensemble_equal":
    return np.stack([
        cb_predict(X),
        models["xgb"].predict(X),
        models["lgbm"].predict(X),
    ], axis=1).mean(axis=1)
  if SELECTED_MODEL == "cb":
    return np.array(cb_predict(X), dtype=float)
  return np.array(models[SELECTED_MODEL].predict(X), dtype=float)

models = load_models(SELECTED_MODEL)

Loading LightGBM model
Loading XGBoost model
Loading CatBoost model


In [84]:
def predict_hypothetical(floor_area_sqm, town, flat_type, remaining_lease_years):
    print(f"Loading model(s): {SELECTED_MODEL}")
    town = town.upper().strip()
    flat_type = flat_type.upper().strip()

    X = build_hypothetical_features(floor_area_sqm, town, flat_type, remaining_lease_years)
    for col in ["town", "flat_type"]:
        X[col] = X[col].astype("category")

    print("Running prediction on hypothetical listing")
    pred_real = predict(models, X) 
    pred_nominal = float(pred_real[0] * (RPI_CURRENT / RPI_BASE))
    print(f"Prediction done. Nominal price: S${pred_nominal:,.0f}")

    # Confidence intervals
    ci_lower = ci_upper = None
    if ci_offsets_json:
        with open(ci_offsets_json) as f:
            all_offsets = json.load(f)
        offsets = all_offsets.get(SELECTED_MODEL) or all_offsets.get("ensemble_equal")
        if offsets:
            ci_lower = float((pred_real[0] + offsets["p025_real"]) * (RPI_CURRENT / RPI_BASE))
            ci_upper = float((pred_real[0] + offsets["p975_real"]) * (RPI_CURRENT / RPI_BASE))
            key_used = SELECTED_MODEL if SELECTED_MODEL in all_offsets else "ensemble_equal"
            print(f"CI offsets applied from key {key_used}: [{ci_lower:,.0f}, {ci_upper:,.0f}]")
    else:
        print("ci_offsets.json not found, CI will be null.")

    return {
        "predicted_price": round(pred_nominal),
        "confidence_low": round(ci_lower) if ci_lower is not None else None,
        "confidence_high": round(ci_upper) if ci_upper is not None else None,
        "town": town,
        "flat_type": flat_type,
        "floor_area_sqm": floor_area_sqm,
        "remaining_lease": remaining_lease_years
    }

In [85]:
# Test
print(predict_hypothetical(144, "WOODLANDS", "EXECUTIVE", 72))

Loading model(s): ensemble_equal
Running prediction on hypothetical listing
Prediction done. Nominal price: S$902,669
CI offsets applied from key ensemble_equal: [776,379, 951,146]
{'predicted_price': 902669, 'confidence_low': 776379, 'confidence_high': 951146, 'town': 'WOODLANDS', 'flat_type': 'EXECUTIVE', 'floor_area_sqm': 144, 'remaining_lease': 72}
